# Unit 3 — Alfred's Gala: Agentic RAG

> **Compatibility:** The smolagents route assembles the original English lessons and their code cells. The CodeAgent uses Python actions, DDGS auto-backend and the Hub's current downloads sort. Actual tool calls are verified. Weather is the course simulation.

Official lessons: [invitees](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/invitees), [tools](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/tools), [agent](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/agent). Choose the smolagents branch, as permitted by the course. The snippets labelled “course reference” are illustrative text from the lessons. Live outputs appear below executable cells.

If DDGS returns no results, the search tool queries the live Wikipedia or Crossref public index.

In [1]:
COURSE_PACKAGES = ['smolagents[openai]==1.26.0', 'datasets==5.2.0', 'langchain-community==0.4.2', 'rank-bm25==0.2.2', 'ddgs==9.16.1']
import os
import subprocess
import sys

if not os.environ.get("HF_COURSE_BATCH"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", *COURSE_PACKAGES], check=True, timeout=600)

In [2]:
import json
import os
from functools import lru_cache

import httpx
from openai import OpenAI

GROQ_BASE = "https://api.groq.com/openai/v1"
QWEN = "qwen/qwen3.8-27b"
GPT_OSS = ("openai/gpt-oss-20b", "openai/gpt-oss-120b")


def optional_secret(name):
    """Read a notebook-authorized secret without printing it."""
    if os.getenv(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return None
    except Exception as exc:
        if type(exc).__name__ in {"SecretNotFoundError", "NotebookAccessError"}:
            return None
        raise


@lru_cache(maxsize=1)
def groq_account():
    key = optional_secret("GROQ_API_KEY")
    if not key:
        raise RuntimeError("Enable notebook access to the GROQ_API_KEY secret in Colab.")
    with OpenAI(api_key=key, base_url=GROQ_BASE, timeout=20, max_retries=0) as client:
        return key, frozenset(model.id for model in client.models.list().data)


class ModelRouting:
    def __init__(self, models, mode):
        self.models = list(models)
        self.mode = mode

    def requests(self, request):
        if request.method != "POST" or not request.url.path.endswith("/chat/completions"):
            yield None, request
            return
        body = json.loads(request.content)
        headers = {key: val for key, val in request.headers.items()
                   if key not in {"content-length", "transfer-encoding"}}
        if self.mode == "code" and body.get("tools"):
            raise ValueError("CodeAgent uses Python actions. Its API request must have no native tools.")
        for model_id in tuple(self.models):
            payload = {**body, "model": model_id}
            max_tokens = payload.pop("max_tokens", None)
            if max_tokens and "max_completion_tokens" not in payload:
                payload["max_completion_tokens"] = max_tokens
            if model_id in GPT_OSS:
                payload.pop("parallel_tool_calls", None)
                payload.pop("stop", None)
                payload["reasoning_effort"] = "low"
            elif model_id == QWEN:
                payload["reasoning_effort"] = "none"
            yield model_id, httpx.Request(
                request.method, request.url, json=payload, headers=headers,
                extensions=request.extensions,
            )

    def accept(self, model_id, status, attempt):
        if model_id is None:
            return True
        if status in {429, 503} and attempt < len(self.models):
            return False
        if 200 <= status < 300 and model_id != self.models[0]:
            self.models.remove(model_id)
            self.models.insert(0, model_id)
            print(f"Groq fallback model: {model_id}", flush=True)
        return True


class CourseTransport(httpx.HTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    def handle_request(self, request):
        import time

        for round_index in range(4):
            for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
                response = super().handle_request(candidate)
                if not self.routing.accept(model_id, response.status_code, attempt):
                    response.close()
                    continue
                if model_id is None or response.status_code not in {429, 503} or round_index == 3:
                    return response
                try:
                    cooldown = float(response.headers.get("retry-after", "0"))
                except ValueError:
                    cooldown = 0
                response.close()
                delay = min(40, max(8 + 5 * round_index, cooldown))
                print(f"Groq is busy; retrying after {delay:.0f} seconds.", flush=True)
                time.sleep(delay)
                break


class AsyncCourseTransport(httpx.AsyncHTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    async def handle_async_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = await super().handle_async_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            await response.aclose()


@lru_cache(maxsize=3)
def groq_route(mode):
    key, available = groq_account()
    candidates = {
        "text": (QWEN, *GPT_OSS),
        "vision": (QWEN,),
        "code": (QWEN, *GPT_OSS),
    }[mode]
    models = [model_id for model_id in candidates if model_id in available]
    if not models:
        raise RuntimeError(f"None of the Groq {mode} models are enabled for this project: {candidates}")
    print(f"Groq {mode} candidates, in order: {', '.join(models)}", flush=True)
    return key, ModelRouting(models, mode)


@lru_cache(maxsize=1)
def make_groq_llm():
    from llama_index.llms.openai_like import OpenAILike
    key, routing = groq_route("text")
    return OpenAILike(
        model=routing.models[0], api_base=GROQ_BASE, api_key=key,
        is_chat_model=True, is_function_calling_model=True, context_window=32768,
        max_tokens=1024, temperature=0.1, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        async_http_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=2)
def make_chat_model(mode="text"):
    from langchain_openai import ChatOpenAI
    key, routing = groq_route(mode)
    return ChatOpenAI(
        model=routing.models[0], base_url=GROQ_BASE, api_key=key,
        temperature=0.1, max_tokens=1024, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        http_async_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=1)
def make_code_model():
    from openai import BadRequestError
    from smolagents import OpenAIModel
    from smolagents.models import ChatMessage

    class PythonActionModel(OpenAIModel):
        def generate(self, *args, **kwargs):
            try:
                return super().generate(*args, **kwargs)
            except BadRequestError as exc:
                body = exc.body if isinstance(exc.body, dict) else {}
                error = body.get("error", body)
                if error.get("code") == "json_validate_failed" and kwargs.get("response_format"):
                    import ast
                    try:
                        value = json.loads(error.get("failed_generation", ""))
                        actions = value if isinstance(value, list) else [value]
                        if not actions or not all(isinstance(action, dict) and isinstance(action.get("code"), str) for action in actions):
                            raise ValueError("Expected Python action objects.")
                        python_code = chr(10).join(action["code"] for action in actions)
                        ast.parse(python_code)
                    except (ValueError, TypeError, SyntaxError):
                        python_code = "print('The action schema was invalid. Generate a single object containing the next Python code action.')"
                    return ChatMessage(role="assistant", content=json.dumps({"code": python_code}))
                if error.get("code") != "tool_use_failed" or not kwargs.get("response_format"):
                    raise
                try:
                    import ast
                    import re
                    raw = error["failed_generation"].strip()
                    try:
                        action = json.loads(raw)
                    except json.JSONDecodeError:
                        legacy = re.fullmatch(r'\{\s*"name"\s*:\s*"python"\s*,\s*"arguments"\s*:\s*(.*?)\s*\}', raw, re.S)
                        if not legacy:
                            assignment = re.fullmatch(r'\{\s*"name"\s*:\s*"(guest_info_retriever|web_search|weather_info|hub_stats|final_answer)"\s*,\s*"arguments"\s*:\s*(.*?)\s*\}', raw, re.S)
                            if not assignment:
                                return ChatMessage(role="assistant", content=json.dumps({"code": "print('Malformed tool request. Regenerate an answer using valid Python code.')"}))
                            tool_name, argument_code = assignment.groups()
                            try:
                                values = {}
                                for statement in ast.parse(argument_code).body:
                                    if not (isinstance(statement, ast.Assign) and len(statement.targets) == 1 and isinstance(statement.targets[0], ast.Name)):
                                        raise ValueError("Expected literal keyword assignments.")
                                    values[statement.targets[0].id] = ast.literal_eval(statement.value)
                                if not values:
                                    raise ValueError("No tool arguments.")
                            except (ValueError, SyntaxError, TypeError):
                                return ChatMessage(role="assistant", content=json.dumps({"code": "print('Malformed tool arguments. Regenerate a valid Python function call.')"}))
                            call = f"{tool_name}(**{values!r})"
                            recovered_code = call if tool_name == "final_answer" else f"print({call})"
                            return ChatMessage(role="assistant", content=json.dumps({"code": recovered_code}))
                        code = legacy.group(1).strip()
                        ast.parse(code)
                        action = {"name": "ThoughtAndCodeAnswer", "arguments": {"code": code}}
                    name = action["name"]
                    arguments = action["arguments"]
                    if name == "python" and isinstance(arguments, str):
                        ast.parse(arguments)
                        name, arguments = "ThoughtAndCodeAnswer", {"code": arguments}
                    elif isinstance(arguments, str):
                        arguments = json.loads(arguments)
                    if not isinstance(arguments, dict):
                        raise ValueError("Expected tool arguments.")
                    if isinstance(arguments.get("code"), str) and name in {"final_answer", "ThoughtAndCodeAnswer"}:
                        action_json = arguments
                    elif name in {"guest_info_retriever", "web_search", "weather_info", "hub_stats", "final_answer"}:
                        call = f"{name}(**{arguments!r})"
                        action_json = {"code": call if name == "final_answer" else f"print({call})"}
                    else:
                        raise ValueError("Unsupported tool name.")
                except (KeyError, ValueError, TypeError):
                    raise exc from None
                print("Groq action decoded from tool_use_failed; executing through CodeAgent.")
                return ChatMessage(role="assistant", content=json.dumps(action_json))

    key, routing = groq_route("code")
    return PythonActionModel(
        model_id=routing.models[0], api_base=GROQ_BASE, api_key=key,
        max_tokens=1800, temperature=0.1, retry=False,
        client_kwargs={
            "http_client": httpx.Client(transport=CourseTransport(routing), timeout=30),
            "timeout": 30, "max_retries": 0,
        },
    )

tool_trace = []

def run_gala(agent, query, expected=(), **kwargs):
    start = len(tool_trace)
    if expected:
        checklist = ", ".join(expected)
        query = f"{query}{chr(10)}For this exercise call each of these tools before your final answer: {checklist}. Use their observed results."
    result = agent.run(query, return_full_result=True, **kwargs)
    missing = [tool for tool in expected if tool not in tool_trace[start:]]
    if missing:
        checklist = ", ".join(missing)
        result = agent.run(f"Continue answering the original question. Call these missing tools now: {checklist}. Then give the final answer using the collected evidence.", return_full_result=True, reset=False)
    called = tool_trace[start:]
    print("Successful tool calls:", called)
    if result.state != "success" or not str(result.output or "").strip():
        raise RuntimeError(f"Gala agent ended with {result.state}.")
    if not set(expected) <= set(called):
        raise RuntimeError(f"Expected tool calls {expected}; observed {called}.")
    return result.output


# Creating a RAG Tool for Guest Stories


Alfred, your trusted agent, is preparing for the most extravagant gala of the century. To ensure the event runs smoothly, Alfred needs quick access to up-to-date information about each guest. Let's help Alfred by creating a custom Retrieval-Augmented Generation (RAG) tool, powered by our custom dataset.

## Why RAG for a Gala?

Imagine Alfred mingling among the guests, needing to recall specific details about each person at a moment's notice. A traditional LLM might struggle with this task because:

1. The guest list is specific to your event and not in the model's training data
2. Guest information may change or be updated frequently
3. Alfred needs to retrieve precise details like email addresses

This is where Retrieval Augmented Generation (RAG) shines! By combining a retrieval system with an LLM, Alfred can access accurate, up-to-date information about your guests on demand.

> [!TIP]
> You can choose any of the frameworks covered in the course for this use case. Select your preferred option from the code tabs.

## Setting up our application

In this unit, we'll develop our agent within a HF Space, as a structured Python project. This approach helps us maintain clean, modular code by organizing different functionalities into separate files.  Also, this makes for a more realistic use case where you would deploy the application for public use.

### Project Structure

- **`tools.py`** – Provides auxiliary tools for the agent.  
- **`retriever.py`** – Implements retrieval functions to support knowledge access.  
- **`app.py`** – Integrates all components into a fully functional agent, which we'll finalize in the last part of this unit.  

For a hands-on reference, check out [this HF Space](https://huggingface.co/spaces/agents-course/Unit_3_Agentic_RAG), where the Agentic RAG developed in this unit is live. Feel free to clone it and experiment!

You can directly test the agent below:

[Interactive course reference](https://agents-course-unit-3-agentic-rag.hf.space)

## Dataset Overview

Our dataset [`agents-course/unit3-invitees`](https://huggingface.co/datasets/agents-course/unit3-invitees/) contains the following fields for each guest:

- **Name**: Guest's full name
- **Relation**: How the guest is related to the host
- **Description**: A brief biography or interesting facts about the guest
- **Email Address**: Contact information for sending invitations or follow-ups

Below is a preview of the dataset:
[Interactive course reference](https://huggingface.co/datasets/agents-course/unit3-invitees/embed/viewer/default/train)

> [!TIP]
> In a real-world scenario, this dataset could be expanded to include dietary preferences, gift interests, conversation topics to avoid, and other helpful details for a host.

## Building the Guestbook Tool

We'll create a custom tool that Alfred can use to quickly retrieve guest information during the gala. Let's break this down into three manageable steps:

1. Load and prepare the dataset
2. Create the Retriever Tool
3. Integrate the Tool with Alfred

Let's start with loading and preparing the dataset!

### Step 1: Load and Prepare the Dataset

First, we need to transform our raw guest data into a format that's optimized for retrieval.



We will use the Hugging Face `datasets` library to load the dataset and convert it into a list of `Document` objects from the `langchain.docstore.document` module.

In [3]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]

In the code above, we:
- Load the dataset
- Convert each guest entry into a `Document` object with formatted content
- Store the `Document` objects in a list

This means we've got all of our data nicely available so we can get started with configuring our retrieval.

### Step 2: Create the Retriever Tool

Now, let's create a custom tool that Alfred can use to search through our guest information.



We will use the `BM25Retriever` from the `langchain_community.retrievers` module to create a retriever tool.

> [!TIP]
> The <code>BM25Retriever</code> is a great starting point for retrieval, but for more advanced semantic search, you might consider using embedding-based retrievers like those from <a href="https://www.sbert.net/">sentence-transformers</a>.

In [4]:
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever

class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {
            "type": "string",
            "description": "The name or relation of the guest you want information about."
        }
    }
    output_type = "string"

    def __init__(self, docs):
        self.is_initialized = False
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            tool_trace.append(self.name)
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No matching guest information found."

# Initialize the tool
guest_info_tool = GuestInfoRetrieverTool(docs)

<ipython-input-1-76d926bc1cfa>:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [5]:
guest_excerpt = guest_info_tool("Lady Ada Lovelace")
assert "Ada Lovelace" in guest_excerpt
print("Guest records:", len(docs))
print(guest_excerpt.split("\n\n")[0])

Guest records: 3
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


Let's understand this tool step-by-step: 
- The `name` and `description` help the agent understand when and how to use this tool
- The `inputs` define what parameters the tool expects (in this case, a search query)
- We're using a `BM25Retriever`, which is a powerful text retrieval algorithm that doesn't require embeddings
- The `forward` method processes the query and returns the most relevant guest information



### Step 3: Integrate the Tool with Alfred

Finally, let's bring everything together by creating our agent and equipping it with our custom tool:

In [6]:
from smolagents import CodeAgent

# Initialize the Groq text completion model
model = make_code_model()

# Create Alfred, our gala agent, with the guest info tool
alfred = CodeAgent(tools=[guest_info_tool], model=model, use_structured_outputs_internally=True, instructions="Use the guest_info_retriever before answering guest questions.", max_steps=4)

# Example query Alfred might receive during the gala
response = run_gala(alfred, "Tell me about our guest named 'Lady Ada Lovelace'.", ("guest_info_retriever",))

print("🎩 Alfred's Response:")
print(response)

Groq code candidates, in order: qwen/qwen3.8-27b, openai/gpt-oss-20b, openai/gpt-oss-120b
╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ Tell me about our guest named 'Lady Ada Lovelace'.                           │
│ For this exercise call each of these tools before your final answer:         │
│ guest_info_retriever. Use their observed results.                            │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Groq action decoded from tool_use_failed; executing through CodeAgent.
 ─ Executing parsed code: ───────────────────────────────────────────────────── 
  info = guest_info_retriever("Lady Ada Lovelace")                              
  print(info)                

Expected output:

```
🎩 Alfred's Response:
Based on the information I retrieved, Lady Ada Lovelace is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine. Her email address is ada.lovelace@example.com.
```

What's happening in this final step:
- We initialize a Hugging Face model using the `InferenceClientModel` class
- We create our agent (Alfred) as a `CodeAgent`, which can execute Python code to solve problems
- We ask Alfred to retrieve information about a guest named "Lady Ada Lovelace"



## Example Interaction

During the gala, a conversation might flow like this:

**You:** "Alfred, who is that gentleman talking to the ambassador?"

**Alfred:** *quickly searches the guest database* "That's Dr. Nikola Tesla, sir. He's an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk."

```json
{
    "name": "Dr. Nikola Tesla",
    "relation": "old friend from university days",  
    "description": "Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.",
    "email": "nikola.tesla@gmail.com"
}
```

## Taking It Further

Now that Alfred can retrieve guest information, consider how you might enhance this system:

1. **Improve the retriever** to use a more sophisticated algorithm like [sentence-transformers](https://www.sbert.net/)
2. **Implement a conversation memory** so Alfred remembers previous interactions
3. **Combine with web search** to get the latest information on unfamiliar guests
4. **Integrate multiple indexes** to get more complete information from verified sources

Now Alfred is fully equipped to handle guest inquiries effortlessly, ensuring your gala is remembered as the most sophisticated and delightful event of the century!

> [!TIP]
> Try extending the retriever tool to also return conversation starters based on each guest's interests or background. How would you modify the tool to accomplish this?
>
> When you're done, implement your guest retriever tool in the <code>retriever.py</code> file.

# Building and Integrating Tools for Your Agent

In this section, we'll grant Alfred access to the web, enabling him to find the latest news and global updates. 
Additionally, he'll have access to weather data and Hugging Face hub model download statistics, so that he can make relevant conversation about fresh topics.

## Give Your Agent Access to the Web

Remember that we want Alfred to establish his presence as a true renaissance host, with a deep knowledge of the world.

To do so, we need to make sure that Alfred has access to the latest news and information about the world.

Let's start by creating a web search tool for Alfred!

In [6]:
from datetime import date
from html import unescape
import re
from urllib.parse import quote

import requests
from ddgs import DDGS
from ddgs.exceptions import DDGSException
from smolagents import Tool

SEARCH_HEADERS = {
    "User-Agent": "HFAgentsCourseColab/1.0 (https://huggingface.co/learn/agents-course; academic notebook)",
    "Accept": "application/json",
}


def _search_json(url, params):
    response = requests.get(url, params=params, headers=SEARCH_HEADERS, timeout=8)
    response.raise_for_status()
    return response.json()


def _wikipedia_results(query):
    # MediaWiki search ranks terms reliably with the interrogative lead-in removed.
    terms = re.sub(r"^(?:who(?:'s| is)|what(?:'s| is)|tell me about)\s+", "", query.strip(), flags=re.I)
    pages = _search_json("https://en.wikipedia.org/w/rest.php/v1/search/page", {"q": terms, "limit": 4})["pages"]
    if not pages and terms != query:
        pages = _search_json("https://en.wikipedia.org/w/rest.php/v1/search/page", {"q": query, "limit": 4})["pages"]
    results = []
    for page in pages:
        key = quote(page["key"], safe="")
        body = unescape(re.sub(r"<[^>]*>", "", page.get("excerpt") or ""))
        if not results:
            try:
                fields = {"action": "query", "prop": "extracts", "exintro": "1",
                          "explaintext": "1", "titles": page["title"], "format": "json", "formatversion": "2"}
                raw = _search_json("https://en.wikipedia.org/w/api.php", fields)["query"]["pages"][0].get("extract", "")
                paragraphs = [p.strip() for p in raw.splitlines() if p.strip()]
                if paragraphs:
                    terms_set = set(re.findall(r"[a-z]{4,}", query.lower()))
                    relevant = max(paragraphs, key=lambda p: (len(set(re.findall(r"[a-z]{4,}", p.lower())) & terms_set),
                                                              "current" in p.lower() if "current" in terms_set else False))
                    if "current" in terms_set:
                        relevant = next((p for p in paragraphs if "current" in p.lower()), relevant)
                    body = paragraphs[0][:250]
                    if relevant != paragraphs[0]:
                        body += " … " + relevant[:280]
            except (requests.RequestException, ValueError, KeyError, IndexError):
                pass
        results.append({"title": page["title"], "href": f"https://en.wikipedia.org/wiki/{key}", "body": body})
    return results


def _crossref_results(query):
    params = {
        "query.bibliographic": query,
        "filter": f"from-pub-date:{date.today().year - 3}-01-01",
        "rows": 4,
        "select": "title,URL,published",
    }
    items = _search_json("https://api.crossref.org/works", params)["message"]["items"]
    return [{"title": item["title"][0], "href": item["URL"],
             "body": f"Publication year: {item.get('published', {}).get('date-parts', [[None]])[0][0]}"}
            for item in items if item.get("title") and item.get("URL")]


class CourseWebSearch(Tool):
    name = "web_search"
    description = "Find current public web sources for a question."
    inputs = {"query": {"type": "string", "description": "The topic to search for."}}
    output_type = "string"

    def forward(self, query: str) -> str:
        try:
            results = DDGS(timeout=8).text(query, max_results=4, backend="auto")
        except DDGSException:
            results = []
        backend = "DDGS"
        if not results:
            technical = bool(re.search(r"wireless|research|paper|technology|advanc", query, flags=re.I))
            services = [("Crossref", _crossref_results), ("Wikipedia", _wikipedia_results)] if technical else [
                ("Wikipedia", _wikipedia_results), ("Crossref", _crossref_results)]
            for backend, service in services:
                try:
                    results = service(query)
                except (requests.RequestException, ValueError, KeyError):
                    continue
                if results:
                    break
        if not results:
            raise RuntimeError("DDGS, Wikipedia and Crossref returned no sources for this query.")
        tool_trace.append(self.name)
        return f"Search backend: {backend}\n" + "\n".join(
            f"{r['title']} — {r['href']} — {r.get('body', '')[:550]}" for r in results
        )


search_tool = CourseWebSearch()
results = search_tool("Who's the current President of France?")
print(results)

Search backend: Wikipedia
President of France — https://en.wikipedia.org/wiki/President_of_France — The president of France (French: Président de la France), officially the president of the French Republic (French: Président de la République française) or president of the Republic (Président de la République), is the executive head of state of Fran … The current president is Emmanuel Macron, who succeeded François Hollande on 14 May 2017 following the 2017 presidential election, and was inaugurated for a second term on 7 May 2022 following the 2022 presidential election.
List of presidents of France — https://en.wikipedia.org/wiki/List_of_presidents_of_France — The president of France is the head of state of France, elected by popular vote for five years. The first officeholder is considered to be Louis-Napoléon
Minister president (Germany) — https://en.wikipedia.org/wiki/Minister_president_%28Germany%29 — The minister-president (German: Ministerpräsident, pronounced [miˈnɪstɐpʁɛziˌdɛn

Expected output (course reference):

```
The current President of France in Emmanuel Macron.
```




## Creating a Custom Tool for Weather Information to Schedule the Fireworks

The perfect gala would have fireworks over a clear sky, we need to make sure the fireworks are not cancelled due to bad weather.

Let's create a custom tool that can be used to call an external weather API and get the weather information for a given location.

> [!TIP]
> For the sake of simplicity, we're using a dummy weather API for this example. If you want to use a real weather API, you could implement a weather tool that uses the OpenWeatherMap API, like in <a href="../../unit1/tutorial">Unit 1</a>.

In [7]:
from smolagents import Tool
import random

class WeatherInfoTool(Tool):
    name = "weather_info"
    description = "Fetches dummy weather information for a given location."
    inputs = {
        "location": {
            "type": "string",
            "description": "The location to get weather information for."
        }
    }
    output_type = "string"

    def forward(self, location: str):
        # Dummy weather data
        weather_conditions = [
            {"condition": "Rainy", "temp_c": 15},
            {"condition": "Clear", "temp_c": 25},
            {"condition": "Windy", "temp_c": 20}
        ]
        # Randomly select a weather condition
        data = random.choice(weather_conditions)
        tool_trace.append(self.name)
        return f"Simulated course weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = WeatherInfoTool()

## Creating a Hub Stats Tool for Influential AI Builders

In attendance at the gala are the who's who of AI builders. Alfred wants to impress them by discussing their most popular models, datasets, and spaces. We'll create a tool to fetch model statistics from the Hugging Face Hub based on a username.

In [8]:
from smolagents import Tool
from huggingface_hub import list_models

class HubStatsTool(Tool):
    name = "hub_stats"
    description = "Fetches the most downloaded model from a specific author on the Hugging Face Hub."
    inputs = {
        "author": {
            "type": "string",
            "description": "The username of the model author/organization to find models from."
        }
    }
    output_type = "string"

    def forward(self, author: str):
        try:
            # List models from the specified author, sorted by downloads
            models = list(list_models(author=author, sort="downloads", limit=1, token=False))
            
            if models:
                model = models[0]
                tool_trace.append(self.name)
                return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
            else:
                raise LookupError(f"No models found for author {author}.")
        except Exception as e:
            raise RuntimeError(f"Hub stats unavailable for {author}.") from e

# Initialize the tool
hub_stats_tool = HubStatsTool()

# Example usage
print(hub_stats_tool("facebook")) # Example: Get the most downloaded model by Facebook

The most downloaded model by facebook is facebook/contriever with 8,301,872 downloads.


Expected output (course reference):

```
The most downloaded model by facebook is facebook/esmfold_v1 with 12,544,550 downloads.
```



With the Hub Stats Tool, Alfred can now impress influential AI builders by discussing their most popular models.

## Integrating Tools with Alfred

Now that we have all the tools, let's integrate them into Alfred's agent:

In [10]:
from smolagents import CodeAgent

# Initialize the Groq text completion model
model = make_code_model()

# Create Alfred with all the tools
alfred = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool], 
    model=model, use_structured_outputs_internally=True, max_steps=5,
    instructions="Use web_search for public background and hub_stats for model download rankings." 
)

# Example query Alfred might receive during the gala
response = run_gala(alfred, "What is Facebook and what's their most popular model?", ("web_search", "hub_stats"))

print("🎩 Alfred's Response:")
print(response)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ What is Facebook and what's their most popular model?                        │
│ For this exercise call each of these tools before your final answer:         │
│ web_search, hub_stats. Use their observed results.                           │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Groq fallback model: openai/gpt-oss-120b
 ─ Executing parsed code: ───────────────────────────────────────────────────── 
  result_fb = web_search(query="Facebook what is it")                           
  print(result_fb)                                                              
 ───────────────────────────────────────────────────────────────────

Expected output: 

```
🎩 Alfred's Response:
Facebook is a social networking website where users can connect, share information, and interact with others. The most downloaded model by Facebook on the Hugging Face Hub is ESMFold_v1.
```



## Conclusion

By integrating these tools, Alfred is now equipped to handle a variety of tasks, from web searches to weather updates and model statistics. This ensures he remains the most informed and engaging host at the gala.

> [!TIP]
> Try implementing a tool that can be used to get the latest news about a specific topic.
>
> When you're done, implement your custom tools in the <code>tools.py</code> file.

# Creating Your Gala Agent

Now that we've built all the necessary components for Alfred, it's time to bring everything together into a complete agent that can help host our extravagant gala. 

In this section, we'll combine the guest information retrieval, web search, weather information, and Hub stats tools into a single powerful agent.

## Assembling Alfred: The Complete Agent

Instead of reimplementing all the tools we've created in previous sections, we'll import them from their respective modules which we saved in the `tools.py` and `retriever.py` files.

> [!TIP]
> If you haven't implemented the tools yet, go back to the <a href="./tools">tools</a> and <a href="./invitees">retriever</a> sections to implement them, and add them to the <code>tools.py</code> and <code>retriever.py</code> files.

Let's import the necessary libraries and tools from the previous sections:

In [9]:
# The tools.py and retriever.py definitions were executed above in this notebook.
from smolagents import CodeAgent

Now, let's combine all these tools into a single agent:

In [10]:
# Initialize the Groq text completion model
model = make_code_model()

# Create Alfred with all the tools
alfred = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool], 
    model=model, use_structured_outputs_internally=True,
    max_steps=6,
    instructions=(
        "For guest details use guest_info_retriever. For weather use weather_info and "
        "make clear that it is simulated. For Hugging Face model rankings use hub_stats. "
        "For recent external information use web_search. For combined requests use all "
        "necessary tools before answering. Cite real search links and use Python actions."
    ),  # Add any additional base tools
    planning_interval=3   # Enable planning every 3 steps
)

Groq code candidates, in order: qwen/qwen3.8-27b, openai/gpt-oss-20b, openai/gpt-oss-120b


Your agent is now ready to use!

## Using Alfred: End-to-End Examples

Now that Alfred is fully equipped with all the necessary tools, let's see how he can help with various tasks during the gala.

### Example 1: Finding Guest Information

Let's see how Alfred can help us with our guest information.

In [13]:
query = "Tell me about 'Lady Ada Lovelace'"
response = run_gala(alfred, query, ('guest_info_retriever',))

print("🎩 Alfred's Response:")
print(response)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ Tell me about 'Lady Ada Lovelace'                                            │
│ For this exercise call each of these tools before your final answer:         │
│ guest_info_retriever. Use their observed results.                            │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
───────────────────────────────── Initial plan ─────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the
task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- The subject to describe is **“Lady Ada Lovelace.”**
- The instruction requires that **the tool `guest_info_retriever` be called** 
(at least once) before giving the final answer, and the observed result fr

Expected output (course reference):

```
🎩 Alfred's Response:
Based on the information I retrieved, Lady Ada Lovelace is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine. Her email address is ada.lovelace@example.com.
```




### Example 2: Checking the Weather for Fireworks

Let's see how Alfred can help us with the weather.

In [14]:
query = "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"
response = run_gala(alfred, query, ('weather_info',))

print("🎩 Alfred's Response:")
print(response)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ What's the weather like in Paris tonight? Will it be suitable for our        │
│ fireworks display?                                                           │
│ For this exercise call each of these tools before your final answer:         │
│ weather_info. Use their observed results.                                    │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
───────────────────────────────── Initial plan ─────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the
task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- The user wants to know the weather in Paris tonight.
- The user wants to know if the weather will be suitable for a fireworks

Expected output (course reference) (will vary due to randomness):
```
🎩 Alfred's Response:
I've checked the weather in Paris for you. Currently, it's clear with a temperature of 25°C. These conditions are perfect for the fireworks display tonight. The clear skies will provide excellent visibility for the spectacular show, and the comfortable temperature will ensure the guests can enjoy the outdoor event without discomfort.
```



### Example 3: Impressing AI Researchers

Let's see how Alfred can help us impress AI researchers.

In [15]:
query = "One of our guests is from Qwen. What can you tell me about their most popular model?"
response = run_gala(alfred, query, ('hub_stats',))

print("🎩 Alfred's Response:")
print(response)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ One of our guests is from Qwen. What can you tell me about their most        │
│ popular model?                                                               │
│ For this exercise call each of these tools before your final answer:         │
│ hub_stats. Use their observed results.                                       │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
───────────────────────────────── Initial plan ─────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the
task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- A guest is from **Qwen**.
- We need information about **their most popular model**.
- The tool **hub_stats** must be called 

Expected output (course reference):

```
🎩 Alfred's Response:
The most popular Qwen model is Qwen/Qwen2.5-VL-7B-Instruct with 3,313,345 downloads.
```


### Example 4: Combining Multiple Tools

Let's see how Alfred can help us prepare for a conversation with Dr. Nikola Tesla.

In [11]:
query = "I need to speak with Dr. Nikola Tesla about recent advancements in wireless energy. Can you help me prepare for this conversation?"
response = run_gala(alfred, query, ('guest_info_retriever', 'web_search'))

print("🎩 Alfred's Response:")
print(response)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ I need to speak with Dr. Nikola Tesla about recent advancements in wireless  │
│ energy. Can you help me prepare for this conversation?                       │
│ For this exercise call each of these tools before your final answer:         │
│ guest_info_retriever, web_search. Use their observed results.                │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
Groq fallback model: openai/gpt-oss-20b
───────────────────────────────── Initial plan ─────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the
task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- The user wants to speak with **Dr. Nikola Tesla** about **recent advancements 
in wir

Expected output (course reference):

```
🎩 Alfred's Response:
I've gathered information to help you prepare for your conversation with Dr. Nikola Tesla.

Guest Information:
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

Recent Advancements in Wireless Energy:
Based on my web search, here are some recent developments in wireless energy transmission:
1. Researchers have made progress in long-range wireless power transmission using focused electromagnetic waves
2. Several companies are developing resonant inductive coupling technologies for consumer electronics
3. There are new applications in electric vehicle charging without physical connections

Conversation Starters:
1. "I'd love to hear about your new patent on wireless energy transmission. How does it compare to your original concepts from our university days?"
2. "Have you seen the recent developments in resonant inductive coupling for consumer electronics? What do you think of their approach?"
3. "How are your pigeons doing? I remember your fascination with them."

This should give you plenty to discuss with Dr. Tesla while demonstrating your knowledge of his interests and recent developments in his field.
```



## Advanced Features: Conversation Memory

To make Alfred even more helpful during the gala, we can enable conversation memory so he remembers previous interactions:

In [12]:
# Create Alfred with conversation memory
alfred_with_memory = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool], 
    model=model, use_structured_outputs_internally=True,
    max_steps=6,
    instructions=(
        "For guest details use guest_info_retriever. For weather use weather_info and "
        "make clear that it is simulated. For Hugging Face model rankings use hub_stats. "
        "For recent external information use web_search. For combined requests use all "
        "necessary tools before answering. Cite real search links and use Python actions."
    ),
    planning_interval=3
)

# First interaction
response1 = run_gala(alfred_with_memory, "Tell me about Lady Ada Lovelace.", ("guest_info_retriever",))
print("🎩 Alfred's First Response:")
print(response1)

# Second interaction (referencing the first)
response2 = run_gala(alfred_with_memory, "What projects is she currently working on?", reset=False)
print("🎩 Alfred's Second Response:")
print(response2)

╭────────────────────────────────── New run ───────────────────────────────────╮
│                                                                              │
│ Tell me about Lady Ada Lovelace.                                             │
│ For this exercise call each of these tools before your final answer:         │
│ guest_info_retriever. Use their observed results.                            │
│                                                                              │
╰─ PythonActionModel - qwen/qwen3.8-27b ───────────────────────────────────────╯
───────────────────────────────── Initial plan ─────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the
task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- The subject to be described is **Lady Ada Lovelace**.
- The answer must be based on information retrieved by calling the tool 
**`guest_info_retriever`** (once or more) before giving the final answer.

###

Notice that none of these three agent approaches directly couple memory with the agent. Is there a specific reason for this design choice 🧐?
* smolagents: Memory is not preserved across different execution runs, you must explicitly state it using `reset=False`.
* LlamaIndex: Requires explicitly adding a context object for memory management within a run.
* LangGraph: Offers options to retrieve previous messages or utilize a dedicated [MemorySaver](https://langchain-ai.github.io/langgraph/tutorials/introduction/#part-3-adding-memory-to-the-chatbot) component.

## Conclusion

Congratulations! You've successfully built Alfred, a sophisticated agent equipped with multiple tools to help host the most extravagant gala of the century. Alfred can now:

1. Retrieve detailed information about guests
2. Check weather conditions for planning outdoor activities
3. Provide insights about influential AI builders and their models
4. Search the web for the latest information
5. Maintain conversation context with memory

With these capabilities, Alfred is ready to ensure your gala is a resounding success, impressing guests with personalized attention and up-to-date information.